<a href="https://colab.research.google.com/github/alee52/LLM_AgenticAI/blob/main/eval_fine_tuned_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/alee52/LLM_AgenticAI/refs/heads/main/data_prep/evaluator.py -O util.py

In [4]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
from util import evaluate

In [5]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "categorize_products"
HF_USER = "leearum95"

LITE_MODE = True

DATA_USER = "leearum95"
DATASET_NAME = f"{DATA_USER}/items_prompts_full"

if LITE_MODE:
  RUN_NAME = "2026-05-04_19.16.07-lite"
  REVISION = None
# else:
#   RUN_NAME = "2025-11-28_18.47.07"
#   REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

In [6]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [7]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

README.md:   0%|          | 0.00/513 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.75M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/753k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/587k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3099 [00:00<?, ? examples/s]

In [8]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [9]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Memory footprint: 2271.0 MB


In [10]:
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [22]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    # generated_ids = output_ids
    return tokenizer.decode(generated_ids)

In [23]:
model_predict(test[0])

['<|begin_of_text|>What is the category of the following product out of the following categories: Media, Arts & Entertainment, Vehicles & Parts, Business & Industrial, Religious & Ceremonial, Bundles, Electronics, Gift Cards, Home & Garden, Baby & Toddler, Health & Beauty, Services, Sporting Goods, Food, Beverages & Tobacco, Hardware, Apparel & Accessories, Toys & Games, Animals & Pet Supplies, Product Add-Ons, Luggage & Bags, Cameras & Optics, Uncategorized, Office Supplies, Software, Furniture\n\nThe Akai Professional APC64 is an advanced Ableton Live MIDI controller with 64 RGB velocity-sensitive pads, touch strips, internal step sequencer, real-time visual feedback, and comprehensive hardware control, designed for music production, live performance, and creative experimentation.\n\nThe category is <|end_of_text|>']

In [24]:
model_predict(test[1])

['<|begin_of_text|>What is the category of the following product out of the following categories: Media, Arts & Entertainment, Vehicles & Parts, Business & Industrial, Religious & Ceremonial, Bundles, Electronics, Gift Cards, Home & Garden, Baby & Toddler, Health & Beauty, Services, Sporting Goods, Food, Beverages & Tobacco, Hardware, Apparel & Accessories, Toys & Games, Animals & Pet Supplies, Product Add-Ons, Luggage & Bags, Cameras & Optics, Uncategorized, Office Supplies, Software, Furniture\n\nThe BikeMaster Heavy Duty Tire Iron is a durable tool designed for easy tire removal and installation on motorcycles and bikes, ensuring reliability for maintenance and repair tasks.\n\nThe category is <|end_of_text|>']

In [14]:
model_predict(test[2])

'<|end_of_text|>'

In [15]:
model_predict(test[10])

'<|end_of_text|>'